# Model Predictions (coupler_NCap_cap_matrix)

## Configuration

In [1]:
# The parameter file is where the hyperparameters are set. 
# It's reccomended to look at that file first, its interesting and you can set stuff there

from parameters import *

## Library

In [2]:
# Disable some console warnings
import os
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf# Disable some console warnings so you can be free of them printing. 
# Comment the next two lines if you are a professional and like looking at warnings.
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices'
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import os, gc
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model

## Dataset

### Load

In [3]:
def load_scaled_split(kind, split):
    # kind: 'one_hot' or 'linear'
    return np.load(f"{DATA_DIR}/npy/{split}_{kind}_encoding_scaled.npy", allow_pickle=True)

if 'Try Both' in ENCODING_TYPE:
    # one-hot
    X_train_one_hot_encoding = load_scaled_split('one_hot', 'x_train')
    X_val_one_hot_encoding   = load_scaled_split('one_hot', 'x_val')
    X_test_one_hot_encoding  = load_scaled_split('one_hot', 'x_test')

    y_train_one_hot_encoding = load_scaled_split('one_hot', 'y_train')
    y_val_one_hot_encoding   = load_scaled_split('one_hot', 'y_val')
    y_test_one_hot_encoding  = load_scaled_split('one_hot', 'y_test')

    # linear
    X_train_linear_encoding = load_scaled_split('linear', 'x_train')
    X_val_linear_encoding   = load_scaled_split('linear', 'x_val')
    X_test_linear_encoding  = load_scaled_split('linear', 'x_test')

    y_train_linear_encoding = load_scaled_split('linear', 'y_train')
    y_val_linear_encoding   = load_scaled_split('linear', 'y_val')
    y_test_linear_encoding  = load_scaled_split('linear', 'y_test')

else:
    if 'one hot' in ENCODING_TYPE:
        kind = 'one_hot'
    elif 'Linear' in ENCODING_TYPE:
        kind = 'linear'
    else:
        raise ValueError(f"Unknown ENCODING_TYPE: {ENCODING_TYPE}")

    X_train = load_scaled_split(kind, 'x_train')
    X_val   = load_scaled_split(kind, 'x_val')
    X_test  = load_scaled_split(kind, 'x_test')

    y_train = load_scaled_split(kind, 'y_train')
    y_val   = load_scaled_split(kind, 'y_val')
    y_test  = load_scaled_split(kind, 'y_test')


### Visualize

In [4]:
# Decide which model file & test set to use
chosen_path = "model/mlp_6_264_464_364_364_864_13_best_model.keras"      
X_test_cur = np.asarray(X_test)
y_test_cur = np.asarray(y_test)
encoding = ENCODING_TYPE.replace(' ','_')
y_encoding_format_name = encoding    

# Load y headers for labeling columns
y_headers_csv = f'y_characteristics_{y_encoding_format_name}_encoding.csv'
with open(y_headers_csv, 'r') as f:
    headers = f.readline().strip().split(',')

In [14]:
#run on CPU
tf.keras.backend.clear_session()
gc.collect()
try:
    tf.config.experimental.reset_memory_stats('GPU:0')
except Exception:
    pass

with tf.device('/CPU:0'):
    chosen_model = load_model(chosen_path, compile=False)  #dont compile it because we just need to predict
    y_pred = chosen_model.predict(X_test_cur, verbose=0)

print(f"\n—— {os.path.basename(chosen_path)} ——")
chosen_model.summary()
print(f"Samples: {len(X_test_cur)} | Targets dim: {y_test_cur.shape[1]}")


—— mlp_6_264_464_364_364_864_13_best_model.keras ——


Model: "functional_16"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input1 (InputLayer) │ (None, 6)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc0 (Dense)         │ (None, 264)       │      1,848 │ input1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_relu0         │ (None, 264)       │          0 │ fc0[0][0]         │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout0 (Dropout)  │ (None, 264)       │          0 │ leaky_relu0[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc1 (Dense)         │ (None, 464)       │    122,960 │ dropout0[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_relu1         │ (None, 464)       │          0 │ fc1[0][0]         │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout1 (Dropout)  │ (None, 464)       │          0 │ leaky_relu1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc2 (Dense)         │ (None, 364)       │    169,260 │ dropout1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_relu2         │ (None, 364)       │          0 │ fc2[0][0]         │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout2 (Dropout)  │ (None, 364)       │          0 │ leaky_relu2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc3 (Dense)         │ (None, 364)       │    132,860 │ dropout2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_relu3         │ (None, 364)       │          0 │ fc3[0][0]         │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout3 (Dropout)  │ (None, 364)       │          0 │ leaky_relu3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc4 (Dense)         │ (None, 864)       │    315,360 │ dropout3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_relu4         │ (None, 864)       │          0 │ fc4[0][0]         │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout4 (Dropout)  │ (None, 864)       │          0 │ leaky_relu4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ continuous (Dense)  │ (None, 3)         │      2,595 │ dropout4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fingers (Dense)     │ (None, 10)        │      8,650 │ dropout4[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 753,533 (2.87 MB)

 Trainable params: 753,533 (2.87 MB)

 Non-trainable params: 0 (0.00 B)

Samples: 65 | Targets dim: 13


# Scaled

In [15]:

N_SAMPLES_TO_SHOW = 50

# combine multi-output predictions into one (N, total_params)
X_test_cur = np.asarray(X_test_cur)
y_test_cur = np.asarray(y_test_cur)

def _to_2d(a, n_rows_expected):
    a = np.asarray(a)
    # (N, P, 1) -> (N, P)
    if a.ndim == 3 and a.shape[-1] == 1:
        a = a[..., 0]
    # (N,) -> (N, 1)
    if a.ndim == 1:
        a = a.reshape(-1, 1)
    # if transposed (P, N) -> (N, P)
    if a.shape[0] != n_rows_expected and a.ndim == 2 and a.shape[1] == n_rows_expected:
        a = a.T
    return a

N, P = y_test_cur.shape  # expected shape for y_pred is (N, P)

# --- normalize y_pred to shape (N, P) ---
if isinstance(y_pred, (list, tuple)):
    heads = [_to_2d(h, N) for h in y_pred]

    # If concatenating all heads matches the target width, do that
    if all(h.shape[0] == N for h in heads) and sum(h.shape[1] for h in heads) == P:
        y_pred = np.concatenate(heads, axis=1)
    else:
        # Otherwise try to find a single head that matches (N,P)
        match = next((h for h in heads if h.shape == (N, P)), None)
        if match is None:
            raise ValueError(
                f"y_test_cur is {(N,P)} but model returned heads {[h.shape for h in heads]}. "
                f"If your true targets are split across heads, you must concatenate them in the same order "
                f"they were created. (Also: don't overwrite y_pred with y_pred[0]/y_pred[1] before this.)"
            )
        y_pred = match
else:
    # y_pred is already an array (single head)
    y_pred = _to_2d(y_pred, N)
    if y_pred.shape != (N, P):
        raise ValueError(
            f"y_pred is {y_pred.shape} but y_test_cur is {(N,P)}. "
            f"This usually means your model is multi-output and you overwrote the list returned by predict "
            f"(e.g., did y_pred = y_pred[1]). Re-run predict and keep the full list."
        )


# --- detect one-hot columns for finger_count and prep decoding ---
fc_base   = "design_options.finger_count"
fc_prefix = fc_base + "_"

fc_idx = [j for j, h in enumerate(headers) if h.startswith(fc_prefix)]
fc_class_values = []
for j in fc_idx:
    # headers like: "design_options.finger_count_4"
    try:
        fc_class_values.append(int(headers[j].split(fc_prefix, 1)[1]))
    except Exception:
        fc_class_values.append(None)

# scaled errors
sq_errors  = (y_test_cur - y_pred) ** 2
abs_errors = np.abs(y_test_cur - y_pred)

# scaled dataframe (exclude finger_count one-hot; add decoded finger_count row)
rows = []
for i in range(n_samples):
    top_to_top       = X_test_cur[i, 0]
    top_to_bottom    = X_test_cur[i, 1]
    top_to_ground    = X_test_cur[i, 2]
    bottom_to_bottom = X_test_cur[i, 3]
    bottom_to_ground = X_test_cur[i, 4]
    ground_to_ground = X_test_cur[i, 5]

    # continuous + non-finger_count one-hot outputs
    for j in range(n_params):
        if j in fc_idx:
            continue
        rows.append({
            "sample_idx": i,
            "top_to_top": top_to_top,
            "top_to_bottom": top_to_bottom,
            "top_to_ground": top_to_ground,
            "bottom_to_bottom": bottom_to_bottom,
            "bottom_to_ground": bottom_to_ground,
            "ground_to_ground": ground_to_ground,
            "param": headers[j],
            "ref":  y_test_cur[i, j],
            "pred": y_pred[i, j],
            "abs_error": abs_errors[i, j],
            "sq_error":  sq_errors[i, j],
        })

    # collapsed finger_count (decode by argmax)
    if len(fc_idx) > 0 and all(v is not None for v in fc_class_values):
        ref_vec  = y_test_cur[i, fc_idx]
        pred_vec = y_pred[i, fc_idx]

        ref_k  = int(np.argmax(ref_vec))
        pred_k = int(np.argmax(pred_vec))

        ref_fc  = int(fc_class_values[ref_k])
        pred_fc = int(fc_class_values[pred_k])

        abs_fc = float(abs(ref_fc - pred_fc))
        sq_fc  = float((ref_fc - pred_fc) ** 2)

        rows.append({
            "sample_idx": i,
            "top_to_top": top_to_top,
            "top_to_bottom": top_to_bottom,
            "top_to_ground": top_to_ground,
            "bottom_to_bottom": bottom_to_bottom,
            "bottom_to_ground": bottom_to_ground,
            "ground_to_ground": ground_to_ground,
            "param": fc_base,
            "ref":  ref_fc,
            "pred": pred_fc,
            "abs_error": abs_fc,
            "sq_error":  sq_fc,
        })

df = pd.DataFrame(rows)

# save scaled predictions
out_csv = Path(f"predictions_and_errors_{y_encoding_format_name}.csv")
df.to_csv(out_csv, index=False, float_format="%.6g")
print(f"\nSaved CSV -> {out_csv.resolve()}\n")

# pretty print per-sample (scaled)
for i in range(n_samples):
    sub = df[df["sample_idx"] == i].copy()
    sub = sub[["param", "ref", "pred", "abs_error", "sq_error"]]
    header_line = (
        f"— Sample {i} — "
        f"X: top_to_top={X_test_cur[i,0]:.9g}, top_to_bottom={X_test_cur[i,1]:.9g}, "
        f"top_to_ground={X_test_cur[i,2]:.9g}, bottom_to_bottom={X_test_cur[i,3]:.9g}, "
        f"bottom_to_ground={X_test_cur[i,4]:.9g}, ground_to_ground={X_test_cur[i,5]:.9g}"
    )
    print(header_line)
    print(sub.to_string(index=False))
    print()

# print global stats (scaled) — exclude finger_count one-hot columns so they don't skew averages
if len(fc_idx) > 0:
    mask = np.ones(n_params, dtype=bool)
    mask[fc_idx] = False
    abs_for_stats = abs_errors[:, mask]
else:
    abs_for_stats = abs_errors

print("Global scaled error stats (excluding finger_count one-hot):")
print("  min abs_error:", float(abs_for_stats.min()))
print("  median abs_error:", float(np.median(abs_for_stats)))
print("  max abs_error:", float(abs_for_stats.max()))
print(
    "\nNote: one-hot/linear encoding can skew averages if included directly. "
    "We exclude finger_count one-hot columns above and instead report the decoded finger_count row.\n"
)



Saved CSV -> /home/olivias/ML_qubit_design/model_predict_coupler_NCap_cap_matrix/predictions_and_errors_one_hot.csv

— Sample 0 — X: top_to_top=0.00832192455, top_to_bottom=0.00565011737, top_to_ground=0.0365676091, bottom_to_bottom=0.00692052511, bottom_to_ground=0.00850165266, ground_to_ground=0.0110262114
                       param  ref     pred  abs_error  sq_error
      design_options.cap_gap  0.0 0.011719   0.011719  0.000137
    design_options.cap_width  0.2 0.107239   0.092761  0.008605
design_options.finger_length  0.0 0.063538   0.063538  0.004037
 design_options.finger_count  1.0 1.000000   0.000000  0.000000

— Sample 1 — X: top_to_top=0.0914761888, top_to_bottom=0.0642074415, top_to_ground=0.158027636, bottom_to_bottom=0.111459797, bottom_to_ground=0.142610781, ground_to_ground=0.146250336
                       param      ref     pred  abs_error  sq_error
      design_options.cap_gap 1.000000 1.046875   0.046875  0.002197
    design_options.cap_width 0.600000 0.518066 

# Unscaled

In [16]:
#unscale everything and look at errors again. 
#You can compare the unscaled actual values to the ml_00...py notebook to convice yourself that unscaling worked

with open('X_names', 'r') as f:
    X_index_names = f.read().splitlines()

# Load headers from the saved numpy file (same ordering as y arrays)
headers = np.load("y_columns.npy", allow_pickle=True).astype(str).tolist()

# --- detect one-hot columns for finger_count and prep decoding ---
fc_base = "design_options.finger_count"
fc_prefix = fc_base + "_"

fc_idx = [j for j, h in enumerate(headers) if h.startswith(fc_prefix)]
fc_class_values = []
for j in fc_idx:
    try:
        fc_class_values.append(int(headers[j].split(fc_prefix, 1)[1]))
    except Exception:
        fc_class_values.append(None)

# unscaling x
X_test_unscaled = np.asarray(X_test_cur.copy())
for i in range(X_test_unscaled.shape[0]):
    for j in range(X_test_unscaled.shape[1]):
        scaler = joblib.load(f'scalers/scaler_X_{y_encoding_format_name}_{X_index_names[j]}.save')
        X_test_unscaled[i, j] = scaler.inverse_transform([[X_test_unscaled[i, j]]])[0][0]

# unscaling y
y_test_unscaled = np.asarray(y_test_cur.copy())
for i in range(y_test_unscaled.shape[0]):
    for j in range(y_test_unscaled.shape[1]):
        if j in fc_idx:
            continue
        scaler = joblib.load(f'scalers/scaler_y_{y_encoding_format_name}_{headers[j]}.save')
        y_test_unscaled[i, j] = scaler.inverse_transform([[y_test_unscaled[i, j]]])[0][0]

# unscaling y predictions
y_pred_unscaled = np.asarray(y_pred.copy())
for i in range(y_pred_unscaled.shape[0]):
    for j in range(y_pred_unscaled.shape[1]):
        if j in fc_idx:
            continue
        scaler = joblib.load(f'scalers/scaler_y_{y_encoding_format_name}_{headers[j]}.save')
        y_pred_unscaled[i, j] = scaler.inverse_transform([[y_pred_unscaled[i, j]]])[0][0]

n_samples, n_params = y_test_unscaled.shape
 

# find how good or bad we did (the errors)
sq_errors_unscaled  = (y_test_unscaled - y_pred_unscaled) ** 2
abs_errors_unscaled = np.abs(y_test_unscaled - y_pred_unscaled)

# making a nice fancy dataframe, we like fancy things
rows_unscaled = []
for i in range(N_SAMPLES_TO_SHOW):
    qubit_frequency_GHz, anharmonicity_MHz = X_test_unscaled[i, 0], X_test_unscaled[i, 1]

    # continuous + non-finger_count one-hot outputs
    for j in range(n_params):
        if j in fc_idx:
            continue
        rows_unscaled.append({
            "sample_idx": i,
            "qubit_frequency_GHz": qubit_frequency_GHz,
            "anharmonicity_MHz": anharmonicity_MHz,
            "param": headers[j],
            "ref_unscaled": y_test_unscaled[i, j],
            "pred_unscaled": y_pred_unscaled[i, j],
            "abs_error_unscaled": abs_errors_unscaled[i, j],
            "sq_error_unscaled": sq_errors_unscaled[i, j],
        })

    # collapsed finger_count (decode by argmax)
    if len(fc_idx) > 0 and all(v is not None for v in fc_class_values):
        ref_vec = y_test_unscaled[i, fc_idx]
        pred_vec = y_pred_unscaled[i, fc_idx]

        ref_k = int(np.argmax(ref_vec))
        pred_k = int(np.argmax(pred_vec))

        ref_fc = int(fc_class_values[ref_k])
        pred_fc = int(fc_class_values[pred_k])

        abs_fc = float(abs(ref_fc - pred_fc))
        sq_fc = float((ref_fc - pred_fc) ** 2)

        rows_unscaled.append({
            "sample_idx": i,
            "qubit_frequency_GHz": qubit_frequency_GHz,
            "anharmonicity_MHz": anharmonicity_MHz,
            "param": fc_base,
            "ref_unscaled": ref_fc,
            "pred_unscaled": pred_fc,
            "abs_error_unscaled": abs_fc,
            "sq_error_unscaled": sq_fc,
        })

df_unscaled = pd.DataFrame(rows_unscaled)

# save csv of unscaled results uncase we lose this notebook due to github blowing up, ya never know
out_csv_unscaled = Path(f"predictions_and_errors_unscaled_{y_encoding_format_name}.csv")
df_unscaled.to_csv(out_csv_unscaled, index=False, float_format="%.6g")
print(f"\nSaved CSV -> {out_csv_unscaled.resolve()}\n")

# print out stuff so you can see it here if you are to lazy like me to open a csv
for i in range(N_SAMPLES_TO_SHOW):
    sub = df_unscaled[df_unscaled["sample_idx"] == i].copy()
    sub = sub[["param", "ref_unscaled", "pred_unscaled", "abs_error_unscaled", "sq_error_unscaled"]]
    header_line = (
        f"— Sample {i} (Unscaled) — "
        f"X: cross_to_ground={X_test_unscaled[i,0]:.9g}, claw_to_ground={X_test_unscaled[i,1]:.9g}, cross_to_claw={X_test_unscaled[i,2]:.9g}, cross_to_cross={X_test_unscaled[i,3]:.9g}, claw_to_claw={X_test_unscaled[i,4]:.9g}, ground_to_ground={X_test_unscaled[i,5]:.9g}"
    )
    print(header_line)
    print(sub.to_string(index=False))
    print()

# look at overall stats, see below comment for a caviat 
# (exclude finger_count one-hot columns from global stats so they don't skew the averages)
if len(fc_idx) > 0:
    mask = np.ones(n_params, dtype=bool)
    mask[fc_idx] = False
    abs_for_stats = abs_errors_unscaled[:, mask]
else:
    abs_for_stats = abs_errors_unscaled

print("Global unscaled error stats:")
print("  min abs_error:", float(abs_for_stats.min()))
print("  median abs_error:", float(np.median(abs_for_stats)))
print("  max abs_error:", float(abs_for_stats.max()))

'''
Here onehot/linear encoding and the MLP which maps categorical data to 1s and 0s is probably 
throwing off the global average. These will be rounded in the future and will probably always 
round to the right number to reconstruct the correct category-- but for now it might throw off 
the overall average error. In the future we might want to just have it consider the non-categorical 
data when finding an overall average and reporting that number.
'''



Saved CSV -> /home/olivias/ML_qubit_design/model_predict_coupler_NCap_cap_matrix/predictions_and_errors_unscaled_one_hot.csv

— Sample 0 (Unscaled) — X: cross_to_ground=14.91395, claw_to_ground=0.6691, cross_to_claw=14.05235, cross_to_cross=13.62255, claw_to_claw=12.78564, ground_to_ground=58.0137
                       param  ref_unscaled  pred_unscaled  abs_error_unscaled  sq_error_unscaled
      design_options.cap_gap      0.000002       0.000002        2.343753e-08       5.493176e-16
    design_options.cap_width      0.000007       0.000006        9.276125e-07       8.604650e-13
design_options.finger_length      0.000016       0.000018        1.906127e-06       3.633321e-12
 design_options.finger_count      1.000000       1.000000        0.000000e+00       0.000000e+00

— Sample 1 (Unscaled) — X: cross_to_ground=21.64499, claw_to_ground=3.83797, cross_to_claw=17.53586, cross_to_cross=27.51404, claw_to_claw=23.29018, ground_to_ground=73.35986
                       param  ref_unsca

'\nHere onehot/linear encoding and the MLP which maps categorical data to 1s and 0s is probably \nthrowing off the global average. These will be rounded in the future and will probably always \nround to the right number to reconstruct the correct category-- but for now it might throw off \nthe overall average error. In the future we might want to just have it consider the non-categorical \ndata when finding an overall average and reporting that number.\n'

In [17]:
m = tf.keras.models.load_model(chosen_path, compile=True)  # loads optimizer too (if saved)

# ---- neurons per layer (your original logic, slightly generalized) ----
neurons_per_layer = [
    l.units for l in m.layers
    if isinstance(l, tf.keras.layers.Dense) and l.name.startswith("fc")
]
print("Best NEURONS_PER_LAYER:", neurons_per_layer)

# ---- dropout rate(s) from Dropout layers ----
dropouts = [
    (l.name, float(l.rate)) for l in m.layers
    if isinstance(l, tf.keras.layers.Dropout)
]
if dropouts:
    print("Best DROPOUT rate(s):", dropouts)
else:
    print("Best DROPOUT rate(s): none found")

# ---- learning rate / schedule details ----
opt = getattr(m, "optimizer", None)
if opt is None:
    print("Best learning rate: (no optimizer found on loaded model)")
else:
    lr_obj = opt.learning_rate  # can be float/Variable OR a schedule

    # Case 1: schedule (e.g., ExponentialDecay)
    if isinstance(lr_obj, tf.keras.optimizers.schedules.LearningRateSchedule):
        print("Best learning rate: (schedule)", type(lr_obj).__name__)

        # Try to print schedule config nicely
        try:
            cfg = lr_obj.get_config()
            print("LR schedule config:")
            for k, v in sorted(cfg.items()):
                print(f"  {k}: {v}")
        except Exception as e:
            print("Could not read LR schedule config:", repr(e))

        # Also print the current LR value at step=optimizer.iterations
        try:
            step = int(tf.keras.backend.get_value(opt.iterations))
            current_lr = float(lr_obj(step).numpy())
            print(f"Current LR at step {step}:", current_lr)
        except Exception as e:
            print("Could not compute current LR from schedule:", repr(e))

    # Case 2: plain numeric / variable LR
    else:
        try:
            lr_val = float(tf.keras.backend.get_value(lr_obj))
            print("Best learning rate:", lr_val)
        except Exception as e:
            print("Best learning rate: (unreadable)", repr(e))


/home/olivias/.local/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 29 variables whereas the saved optimizer has 33 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Best NEURONS_PER_LAYER: [264, 464, 364, 364, 864]
Best DROPOUT rate(s): [('dropout0', 0.06), ('dropout1', 0.06), ('dropout2', 0.06), ('dropout3', 0.06), ('dropout4', 0.06)]
Best learning rate: 0.0009287500288337469
